# 예제 05. 자동미분
빅데이터프로그래밍 · 4주차

## 목표
- `requires_grad` 와 `backward()` 의 관계를 이해한다
- 손으로 구한 도함수와 결과를 비교한다
- 기울기가 누적된다는 점을 확인한다
- 경사하강법을 한 번 돌려 본다

자동미분은 "어떤 값이 변할 때 결과가 얼마나 변하는지"를 계산하는 기능입니다.


In [ ]:
import torch


## 1. 가장 간단한 예
$y = x^2$ 의 도함수는 $2x$ 입니다. $x=3$ 이면 6.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()

print("x.grad =", x.grad)      # 6.0


## 2. 손으로 구한 값과 비교
$y = 3x^2 + 2x + 1$ → $dy/dx = 6x + 2$. $x=2$ 이면 14.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = 3 * x**2 + 2 * x + 1
y.backward()

print("자동미분 :", x.grad.item())
print("손 계산  :", 6 * 2 + 2)


## 3. 변수가 두 개일 때 — 편미분
$z = x^2 y + y^3$ 에서
- $\partial z/\partial x = 2xy$
- $\partial z/\partial y = x^2 + 3y^2$


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

z = x**2 * y + y**3
z.backward()

print("dz/dx =", x.grad.item(), " (손 계산:", 2*2*3, ")")
print("dz/dy =", y.grad.item(), " (손 계산:", 2**2 + 3*3**2, ")")


## 4. 기울기는 누적됩니다
`backward()` 를 두 번 부르면 값이 더해집니다. 학습 루프에서 매번 초기화해야 하는 이유입니다.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)

y = x ** 2
y.backward(retain_graph=True)
print("1회:", x.grad.item())

y.backward()
print("2회:", x.grad.item(), "← 6이 아니라 12")

x.grad.zero_()
print("초기화 후:", x.grad.item())


## 5. 벡터에 대한 미분
손실은 보통 스칼라 하나입니다. 그래서 마지막에 `mean()` 이나 `sum()` 을 붙입니다.


In [ ]:
w = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
loss = (w ** 2).sum()
loss.backward()

print(w.grad)      # 2w = [2, 4, 6]


## 6. 경사하강법 한 바퀴
$y = (x-4)^2$ 의 최소는 $x=4$ 입니다. 기울기의 반대 방향으로 조금씩 움직입니다.


In [ ]:
x = torch.tensor(0.0, requires_grad=True)
lr = 0.1

for step in range(20):
    y = (x - 4) ** 2
    y.backward()

    with torch.no_grad():        # 갱신 자체는 미분 대상이 아닙니다
        x -= lr * x.grad

    x.grad.zero_()

    if step % 4 == 0 or step == 19:
        print(f"step {step:2d}  x = {x.item():.4f}  y = {y.item():.4f}")


## 7. 미분을 끄는 두 가지 방법
예측만 할 때는 기울기가 필요 없습니다. 메모리와 시간을 아낍니다.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)

with torch.no_grad():
    y = x ** 2
print("no_grad 안:", y.requires_grad)

y2 = (x ** 2).detach()
print("detach 후 :", y2.requires_grad)


## 직접 해보기
1. $y = x^3$ 에서 $x=2$ 일 때의 도함수를 자동미분으로 구하고 손 계산과 비교하세요. (정답 12)
2. $(x-7)^2$ 의 최소를 경사하강법으로 찾으세요.
3. 학습률을 1.0으로 바꾸면 어떻게 되는지 확인하세요.


In [ ]:
# 여기에 작성하세요
